# Supplementary Tables: Ensembl × CAT Gene Concordance

Generates two supplementary TSV tables for publication.

| Table | File | Content |
|-------|------|---------|
| S1 | supp_table_s1_gene_concordance_summary.tsv | Per-gene summary across all 462 assemblies |
| S2 | supp_table_s2_per_assembly_gene_pairs.tsv | Per-assembly × gene-pair detail |

**Input:** Pipeline output directory (set OUTPUT_DIR below)

In [ ]:
!pip install -U pandas pyarrow

In [1]:
import os
import time
import pandas as pd
import numpy as np
from pathlib import Path
from collections import Counter
import warnings
warnings.filterwarnings('ignore')

OUTPUT_DIR = Path(os.getenv('HPRC_QC_OUTPUT_DIR',
    '/hps/nobackup/flicek/ensembl/genebuild/jackt/hprc/hprc-qc/results'))

QC_DIR = OUTPUT_DIR / 'qc_metrics'
RESULTS_DIR = OUTPUT_DIR / 'results'
SUPP_DIR = OUTPUT_DIR / 'supplementary_tables'
SUPP_DIR.mkdir(parents=True, exist_ok=True)

# Parquet caches — dramatically faster than re-reading 462 TSV files each run.
# Set FORCE_RELOAD=True to regenerate from raw files (e.g. after pipeline re-run).
CACHE_DIR = SUPP_DIR / 'cache'
CACHE_DIR.mkdir(parents=True, exist_ok=True)
FORCE_RELOAD = False

print(f"OUTPUT_DIR: {OUTPUT_DIR}")
print(f"QC_DIR exists: {QC_DIR.exists()}")
print(f"RESULTS_DIR exists: {RESULTS_DIR.exists()}")
print(f"CACHE_DIR: {CACHE_DIR}")
print(f"FORCE_RELOAD: {FORCE_RELOAD}")

OSError: [Errno 30] Read-only file system: '/hps'

## Load per-assembly data

In [ ]:
_cache = CACHE_DIR / 'transcript_concordance.parquet'
if _cache.exists() and not FORCE_RELOAD:
    t0 = time.time()
    transcript_concordance_df = pd.read_parquet(_cache)
    print(f"Loaded transcript concordance from cache: {len(transcript_concordance_df):,} rows ({time.time()-t0:.1f}s)")
else:
    transcript_concordance_frames = []
    for i, accession_dir in enumerate(sorted(QC_DIR.iterdir())):
        if not accession_dir.is_dir():
            continue
        accession = accession_dir.name
        fp = accession_dir / f'{accession}_transcript_concordance.tsv'
        try:
            df = pd.read_csv(fp, sep='\t')
            if df.empty:
                print(f"WARNING: empty file {fp}")
                continue
            transcript_concordance_frames.append(df)
        except FileNotFoundError:
            print(f"WARNING: missing file {fp}")
        except Exception as e:
            print(f"WARNING: could not read {fp}: {e}")
        if (i + 1) % 50 == 0:
            print(f"  transcript concordance: {i+1} assemblies loaded...")

    transcript_concordance_df = pd.concat(transcript_concordance_frames, ignore_index=True)
    transcript_concordance_df.to_parquet(_cache, index=False)
    print(f"Loaded transcript concordance: {len(transcript_concordance_df):,} rows from {len(transcript_concordance_frames)} assemblies → cached")

In [ ]:
_cache = CACHE_DIR / 'coding_integrity.parquet'
if _cache.exists() and not FORCE_RELOAD:
    t0 = time.time()
    coding_integrity_df = pd.read_parquet(_cache)
    print(f"Loaded coding integrity from cache: {len(coding_integrity_df):,} rows ({time.time()-t0:.1f}s)")
else:
    coding_integrity_frames = []
    for i, accession_dir in enumerate(sorted(QC_DIR.iterdir())):
        if not accession_dir.is_dir():
            continue
        accession = accession_dir.name
        fp = accession_dir / f'{accession}_coding_integrity.tsv'
        try:
            df = pd.read_csv(fp, sep='\t')
            if df.empty:
                print(f"WARNING: empty file {fp}")
                continue
            coding_integrity_frames.append(df)
        except FileNotFoundError:
            print(f"WARNING: missing file {fp}")
        except Exception as e:
            print(f"WARNING: could not read {fp}: {e}")
        if (i + 1) % 50 == 0:
            print(f"  coding integrity: {i+1} assemblies loaded...")

    coding_integrity_df = pd.concat(coding_integrity_frames, ignore_index=True)
    coding_integrity_df.to_parquet(_cache, index=False)
    print(f"Loaded coding integrity: {len(coding_integrity_df):,} rows from {len(coding_integrity_frames)} assemblies → cached")

In [ ]:
import gc

def safe_mode(series):
    m = series.dropna().mode()
    return m.iloc[0] if len(m) > 0 else np.nan

# Gene presence files are skipped entirely — all IDs in the pipeline output are
# assembly-specific, so the only canonical gene key is ensembl_name (HGNC symbol)
# from the rbh files. gp_agg, tc_medians, and rbh_name_lookup are all built in
# cell-8 from rbh_df and merged, while those DataFrames are already in memory.
print("Helpers defined.")

In [ ]:
_cache = CACHE_DIR / 'divergence.parquet'
if _cache.exists() and not FORCE_RELOAD:
    t0 = time.time()
    divergence_df = pd.read_parquet(_cache)
    print(f"Loaded grch38 divergence from cache: {len(divergence_df):,} rows ({time.time()-t0:.1f}s)")
else:
    divergence_frames = []
    for i, accession_dir in enumerate(sorted(QC_DIR.iterdir())):
        if not accession_dir.is_dir():
            continue
        accession = accession_dir.name
        fp = accession_dir / f'{accession}_grch38_divergence.tsv'
        if not fp.exists():
            continue
        try:
            df = pd.read_csv(fp, sep='\t')
            if df.empty:
                print(f"WARNING: empty file {fp}")
                continue
            divergence_frames.append(df)
        except Exception as e:
            print(f"WARNING: could not read {fp}: {e}")
        if (i + 1) % 50 == 0:
            print(f"  divergence: {len(divergence_frames)} assemblies loaded...")

    if divergence_frames:
        divergence_df = pd.concat(divergence_frames, ignore_index=True)
        divergence_df.to_parquet(_cache, index=False)
        print(f"Loaded grch38 divergence: {len(divergence_df):,} rows from {len(divergence_frames)} assemblies → cached")
    else:
        divergence_df = pd.DataFrame(columns=['assembly_accession', 'sample_name', 'ensembl_gene_id',
                                               'cat_gene_id', 'gene_name', 'ensembl_biotype',
                                               'ref_biotype', 'divergence_category'])
        print("WARNING: no grch38 divergence files found; divergence_df is empty")

## Build Supplementary Table S2: Per-assembly gene-pair detail

In [ ]:
t0 = time.time()

# Load all RBH gene pair files.
# rbh files use ensembl_id/cat_id and have no assembly_accession —
# inject it from the directory name and rename to pipeline-wide conventions.
rbh_frames = []
for i, accession_dir in enumerate(sorted(RESULTS_DIR.iterdir())):
    if not accession_dir.is_dir():
        continue
    accession = accession_dir.name
    fp = accession_dir / f'{accession}.gene_pairs_rbh.tsv'
    try:
        df = pd.read_csv(fp, sep='\t')
        if df.empty:
            print(f"WARNING: empty file {fp}")
            continue
        df['assembly_accession'] = accession
        df = df.rename(columns={'ensembl_id': 'ensembl_gene_id', 'cat_id': 'cat_gene_id'})
        rbh_frames.append(df)
    except FileNotFoundError:
        print(f"WARNING: missing file {fp}")
    except Exception as e:
        print(f"WARNING: could not read {fp}: {e}")
    if (i + 1) % 50 == 0:
        print(f"  RBH pairs: {len(rbh_frames)} assemblies loaded...")

rbh_df = pd.concat(rbh_frames, ignore_index=True)
del rbh_frames
gc.collect()
print(f"Loaded RBH gene pairs: {len(rbh_df):,} rows from {rbh_df['assembly_accession'].nunique()} assemblies ({time.time()-t0:.1f}s)")

# Keep only RBH pairs (all rows should already have is_rbh=True, but filter to be safe)
rbh_df = rbh_df[rbh_df['is_rbh'] == True].copy()
print(f"After filtering is_rbh=True: {len(rbh_df):,} rows")

# --- Build S2_full: per-assembly gene-pair detail (data deposit, not journal supplement) ---

join_cols = ['assembly_accession', 'ensembl_gene_id', 'cat_gene_id']

# Join transcript concordance
tc_cols = join_cols + [
    'n_ensembl_transcripts', 'n_cat_transcripts',
    'n_ens_exact', 'n_cat_exact',
    'ens_to_cat_concordance_rate', 'cat_to_ens_concordance_rate',
    'avg_jaccard_index',
]
tc_cols = [c for c in tc_cols if c in transcript_concordance_df.columns]
tc_subset = transcript_concordance_df[tc_cols].drop_duplicates(subset=join_cols)
merged = rbh_df.merge(tc_subset, on=join_cols, how='left')
print(f"After joining transcript concordance: {len(merged):,} rows")

# Join coding integrity
ci_cols = join_cols + ['classification', 'start_codon_match', 'stop_codon_match', 'frameshift_detected']
ci_cols = [c for c in ci_cols if c in coding_integrity_df.columns]
ci_subset = coding_integrity_df[ci_cols].drop_duplicates(subset=join_cols)
ci_subset = ci_subset.rename(columns={'classification': 'cds_classification'})
merged = merged.merge(ci_subset, on=join_cols, how='left')
print(f"After joining coding integrity: {len(merged):,} rows")

# Join divergence
if not divergence_df.empty:
    div_cols = [c for c in join_cols + ['divergence_category'] if c in divergence_df.columns]
    div_subset = divergence_df[div_cols].drop_duplicates(subset=join_cols)
    merged = merged.merge(div_subset, on=join_cols, how='left')
else:
    merged['divergence_category'] = None
print(f"After joining divergence: {len(merged):,} rows")

# Compute exact-match pct columns
merged['ens_to_cat_exact_pct'] = np.where(
    merged['n_ensembl_transcripts'] > 0,
    (merged['n_ens_exact'] / merged['n_ensembl_transcripts'] * 100).round(1),
    np.nan
)
merged['cat_to_ens_exact_pct'] = np.where(
    merged['n_cat_transcripts'] > 0,
    (merged['n_cat_exact'] / merged['n_cat_transcripts'] * 100).round(1),
    np.nan
)

s2_full_cols = [
    'assembly_accession', 'ensembl_gene_id', 'cat_gene_id',
    'ensembl_name', 'cat_name', 'ensembl_biotype', 'cat_biotype',
    'frac_ensembl_covered', 'frac_cat_covered',
    'n_ensembl_transcripts', 'n_cat_transcripts',
    'n_ens_exact', 'n_cat_exact',
    'ens_to_cat_exact_pct', 'cat_to_ens_exact_pct',
    'ens_to_cat_concordance_rate', 'cat_to_ens_concordance_rate',
    'avg_jaccard_index',
    'cds_classification', 'start_codon_match', 'stop_codon_match', 'frameshift_detected',
    'divergence_category',
]
s2_full_cols = [c for c in s2_full_cols if c in merged.columns]
s2_full_df = merged[s2_full_cols].copy()
out_s2_full = SUPP_DIR / 'supp_table_s2_full_gene_pairs.tsv'
s2_full_df.to_csv(out_s2_full, sep='\t', index=False)
print(f"\nSaved S2 full data deposit: {out_s2_full}")
print(f"S2 full shape: {s2_full_df.shape} ({time.time()-t0:.1f}s)")
del s2_full_df

# --- Pre-compute S1 aggregates while rbh_df and merged are in memory ---
# ensembl_name is the HGNC symbol — the only canonical cross-assembly gene key.

n_assemblies_total = rbh_df['assembly_accession'].nunique()

# gp_agg: one row per canonical gene — how many assemblies have it as an RBH pair
gp_agg = (
    rbh_df.groupby('ensembl_name', sort=False)
    .agg(
        n_assemblies_both=('assembly_accession', 'nunique'),
        ensembl_biotype=('ensembl_biotype', safe_mode),
    )
    .reset_index()
    .rename(columns={'ensembl_name': 'gene_name'})
)
gp_agg['n_assemblies_assessed'] = gp_agg['n_assemblies_both']
gp_agg['pct_assemblies_both'] = (gp_agg['n_assemblies_both'] / n_assemblies_total * 100).round(1)
print(f"gp_agg: {len(gp_agg):,} canonical genes ({time.time()-t0:.1f}s)")

# tc_medians: median transcript concordance per canonical gene
tc_medians = (
    merged.dropna(subset=['ensembl_name'])
    .groupby('ensembl_name')
    .agg(
        median_ens_to_cat_exact_pct=('ens_to_cat_exact_pct', 'median'),
        median_cat_to_ens_exact_pct=('cat_to_ens_exact_pct', 'median'),
        median_ens_to_cat_concordance_rate=('ens_to_cat_concordance_rate', 'median'),
        median_cat_to_ens_concordance_rate=('cat_to_ens_concordance_rate', 'median'),
    )
    .reset_index().round(1)
    .rename(columns={'ensembl_name': 'gene_name'})
)
print(f"tc_medians: {len(tc_medians):,} genes ({time.time()-t0:.1f}s)")

# rbh_name_lookup: (assembly_accession, ensembl_gene_id) → gene_name for CI join in cell-10
rbh_name_lookup = (
    rbh_df[['assembly_accession', 'ensembl_gene_id', 'ensembl_name']]
    .drop_duplicates(['assembly_accession', 'ensembl_gene_id'])
    .rename(columns={'ensembl_name': 'gene_name'})
    .reset_index(drop=True)
)
print(f"rbh_name_lookup: {len(rbh_name_lookup):,} entries ({time.time()-t0:.1f}s)")

# Per-assembly base stats — n_pairs and concordance rates, computed while in memory
assm_n_pairs = rbh_df.groupby('assembly_accession').agg(
    n_rbh_gene_pairs=('ensembl_gene_id', 'count'),
    n_protein_coding_rbh=('ensembl_biotype', lambda x: (x == 'protein_coding').sum()),
).reset_index()
assm_tc = (
    merged.groupby('assembly_accession').agg(
        median_ens_to_cat_concordance_rate=('ens_to_cat_concordance_rate', 'median'),
        median_cat_to_ens_concordance_rate=('cat_to_ens_concordance_rate', 'median'),
    ).reset_index().round(3)
)
assm_summary = assm_n_pairs.merge(assm_tc, on='assembly_accession', how='left')
print(f"assm_summary base: {len(assm_summary):,} assemblies ({time.time()-t0:.1f}s)")

del merged, rbh_df, transcript_concordance_df
gc.collect()
print("Large DataFrames freed.")

## Build Supplementary Table S1: Gene-level concordance summary

In [ ]:
t0 = time.time()

# Work with a slim view of divergence_df — only the columns we need
div_slim = divergence_df[[
    'assembly_accession', 'gene_name', 'is_coding',
    'ens_cds_match', 'cat_cds_match',
    'ens_cds_change_type', 'cat_cds_change_type',
    'divergence_category',
]].copy()

# Normalise cds_match booleans (parquet may store as object True/False/None)
div_slim['ens_cds_match'] = div_slim['ens_cds_match'].map({True: 1.0, False: 0.0})
div_slim['cat_cds_match'] = div_slim['cat_cds_match'].map({True: 1.0, False: 0.0})
print(f"div_slim ready: {len(div_slim):,} rows ({time.time()-t0:.1f}s)")

# ── 1. CDS integrity per canonical gene (Ensembl vs CAT agreement on assembly) ──
ci_with_gene = coding_integrity_df.merge(
    rbh_name_lookup, on=['assembly_accession', 'ensembl_gene_id'], how='inner'
)
print(f"ci_with_gene: {len(ci_with_gene):,} rows ({time.time()-t0:.1f}s)")

for col in ['start_codon_match', 'stop_codon_match', 'frameshift_detected']:
    if col in ci_with_gene.columns:
        ci_with_gene[col] = ci_with_gene[col].map(
            {True: True, False: False, 'True': True, 'False': False, 1: True, 0: False}
        )

ci_agg = ci_with_gene.groupby('gene_name').agg(
    n_assemblies_cds_assessed=('assembly_accession', 'nunique'),
    _n_start=('start_codon_match', 'count'), _sum_start=('start_codon_match', 'sum'),
    _n_stop=('stop_codon_match', 'count'),   _sum_stop=('stop_codon_match', 'sum'),
    _n_fs=('frameshift_detected', 'count'),  _sum_fs=('frameshift_detected', 'sum'),
).reset_index()
del ci_with_gene
gc.collect()

for col, n, s in [('pct_start_codon_match', '_n_start', '_sum_start'),
                   ('pct_stop_codon_match',  '_n_stop',  '_sum_stop'),
                   ('pct_frameshift_detected', '_n_fs',  '_sum_fs')]:
    ci_agg[col] = np.where(ci_agg[n] > 0, (ci_agg[s] / ci_agg[n] * 100).round(1), np.nan)
ci_agg = ci_agg[['gene_name', 'n_assemblies_cds_assessed',
                   'pct_start_codon_match', 'pct_stop_codon_match', 'pct_frameshift_detected']]
print(f"ci_agg: {len(ci_agg):,} genes ({time.time()-t0:.1f}s)")

# ── 2. Divergence category pcts per canonical gene (vs GRCh38, Ensembl vs CAT) ──
DIV_CATS = ['both_agree_reference', 'both_agree_diverged',
            'cat_specific_divergence', 'ensembl_specific_divergence', 'insufficient_data']

div_base = div_slim.dropna(subset=['gene_name', 'divergence_category'])
div_counts = (
    div_base.groupby(['gene_name', 'divergence_category'])
    .size().unstack(fill_value=0).reset_index()
)
for cat in DIV_CATS:
    if cat not in div_counts.columns:
        div_counts[cat] = 0
div_counts['_n_div_total'] = div_counts[DIV_CATS].sum(axis=1)
for cat in DIV_CATS:
    div_counts[f'pct_{cat}'] = np.where(
        div_counts['_n_div_total'] > 0,
        (div_counts[cat] / div_counts['_n_div_total'] * 100).round(1), np.nan)
div_mode = (
    div_base.groupby('gene_name')['divergence_category']
    .agg(safe_mode).reset_index()
)
div_mode.columns = ['gene_name', 'predominant_divergence_category']
pct_div_cols = [f'pct_{cat}' for cat in DIV_CATS]
div_final = div_counts[['gene_name'] + pct_div_cols].merge(div_mode, on='gene_name', how='left')
print(f"div_final: {len(div_final):,} genes ({time.time()-t0:.1f}s)")

# ── 3. Reference-anchored CDS quality per gene (each annotator vs GRCh38) ──
# Filter to coding reference genes — non-coding have ens_cds_change_type='non_coding'
div_coding = div_slim[div_slim['is_coding'] == True].dropna(subset=['gene_name']).copy()

# Binary: does each annotator's CDS match GRCh38?
ref_cds_agg = div_coding.groupby('gene_name').agg(
    _n_ens=('ens_cds_match', 'count'), _sum_ens=('ens_cds_match', 'sum'),
    _n_cat=('cat_cds_match', 'count'), _sum_cat=('cat_cds_match', 'sum'),
).reset_index()
ref_cds_agg['pct_ens_cds_match_ref'] = np.where(
    ref_cds_agg['_n_ens'] > 0,
    (ref_cds_agg['_sum_ens'] / ref_cds_agg['_n_ens'] * 100).round(1), np.nan)
ref_cds_agg['pct_cat_cds_match_ref'] = np.where(
    ref_cds_agg['_n_cat'] > 0,
    (ref_cds_agg['_sum_cat'] / ref_cds_agg['_n_cat'] * 100).round(1), np.nan)

# Specific failure modes: coding_lost and frameshift per annotator
div_coding['ens_coding_lost']    = div_coding['ens_cds_change_type'] == 'coding_lost'
div_coding['cat_coding_lost']    = div_coding['cat_cds_change_type'] == 'coding_lost'
div_coding['ens_frameshift_ref'] = div_coding['ens_cds_change_type'].isin(
    ['frameshift_shorter', 'frameshift_longer'])
div_coding['cat_frameshift_ref'] = div_coding['cat_cds_change_type'].isin(
    ['frameshift_shorter', 'frameshift_longer'])

cds_fail_agg = div_coding.groupby('gene_name').agg(
    _n=('ens_cds_change_type', 'count'),
    _sum_ens_cl=('ens_coding_lost', 'sum'),    _sum_cat_cl=('cat_coding_lost', 'sum'),
    _sum_ens_fs=('ens_frameshift_ref', 'sum'), _sum_cat_fs=('cat_frameshift_ref', 'sum'),
).reset_index()
for col, s in [('pct_ens_coding_lost', '_sum_ens_cl'), ('pct_cat_coding_lost', '_sum_cat_cl'),
               ('pct_ens_frameshift_ref', '_sum_ens_fs'), ('pct_cat_frameshift_ref', '_sum_cat_fs')]:
    cds_fail_agg[col] = np.where(
        cds_fail_agg['_n'] > 0, (cds_fail_agg[s] / cds_fail_agg['_n'] * 100).round(1), np.nan)

# Most common CDS change type per annotator per gene
ens_type_mode = (div_coding.dropna(subset=['ens_cds_change_type'])
    .groupby('gene_name')['ens_cds_change_type'].agg(safe_mode).reset_index()
    .rename(columns={'ens_cds_change_type': 'predominant_ens_cds_change_type'}))
cat_type_mode = (div_coding.dropna(subset=['cat_cds_change_type'])
    .groupby('gene_name')['cat_cds_change_type'].agg(safe_mode).reset_index()
    .rename(columns={'cat_cds_change_type': 'predominant_cat_cds_change_type'}))
del div_coding
gc.collect()

ref_cds_agg = (
    ref_cds_agg[['gene_name', 'pct_ens_cds_match_ref', 'pct_cat_cds_match_ref']]
    .merge(cds_fail_agg[['gene_name', 'pct_ens_coding_lost', 'pct_cat_coding_lost',
                          'pct_ens_frameshift_ref', 'pct_cat_frameshift_ref']], on='gene_name', how='left')
    .merge(ens_type_mode, on='gene_name', how='left')
    .merge(cat_type_mode, on='gene_name', how='left')
)
del cds_fail_agg, ens_type_mode, cat_type_mode
gc.collect()
print(f"ref_cds_agg: {len(ref_cds_agg):,} coding genes ({time.time()-t0:.1f}s)")

# ── 4. S2: Per-assembly summary (462 rows — the actual supplement) ──
# CDS full-match rate per assembly (exclude No_CDS rows from denominator)
ci_cls = coding_integrity_df.groupby(['assembly_accession', 'classification']).size().unstack(fill_value=0).reset_index()
ci_cls.columns.name = None
ci_class_cols = [c for c in ci_cls.columns if c != 'assembly_accession']
ci_with_cds = [c for c in ci_class_cols if c != 'No_CDS']
ci_cls['_n_with_cds'] = ci_cls[[c for c in ci_with_cds]].sum(axis=1)
ci_cls['pct_full_cds_match'] = np.where(
    ci_cls['_n_with_cds'] > 0,
    (ci_cls.get('Full_Match', 0) / ci_cls['_n_with_cds'] * 100).round(1), np.nan)
assm_ci = ci_cls[['assembly_accession', 'pct_full_cds_match']]
del ci_cls
gc.collect()

# Divergence category pcts per assembly
assm_div = (
    div_slim.dropna(subset=['divergence_category'])
    .groupby(['assembly_accession', 'divergence_category'])
    .size().unstack(fill_value=0).reset_index()
)
assm_div.columns.name = None
for cat in DIV_CATS:
    if cat not in assm_div.columns:
        assm_div[cat] = 0
assm_div['_n_div_total'] = assm_div[DIV_CATS].sum(axis=1)
for cat in DIV_CATS:
    assm_div[f'pct_{cat}'] = np.where(
        assm_div['_n_div_total'] > 0,
        (assm_div[cat] / assm_div['_n_div_total'] * 100).round(1), np.nan)
assm_div = assm_div[['assembly_accession'] + [f'pct_{cat}' for cat in DIV_CATS]]

del div_slim
gc.collect()

assm_summary = (
    assm_summary
    .merge(assm_ci,  on='assembly_accession', how='left')
    .merge(assm_div, on='assembly_accession', how='left')
    .sort_values('assembly_accession').reset_index(drop=True)
)
del assm_ci, assm_div
gc.collect()

out_s2 = SUPP_DIR / 'supp_table_s2_per_assembly_summary.tsv'
assm_summary.to_csv(out_s2, sep='\t', index=False)
print(f"Saved S2: {out_s2}")
print(f"S2 shape: {assm_summary.shape} ({time.time()-t0:.1f}s)")

# ── 5. Assemble S1 (enriched) ──
s1_df = gp_agg.copy()
s1_df = s1_df.merge(tc_medians,  on='gene_name', how='left')
s1_df = s1_df.merge(ci_agg,      on='gene_name', how='left')
s1_df = s1_df.merge(div_final,   on='gene_name', how='left')
s1_df = s1_df.merge(ref_cds_agg, on='gene_name', how='left')
s1_df['predominant_divergence_category'] = s1_df['predominant_divergence_category'].fillna('N/A')

s1_col_order = [
    'gene_name', 'ensembl_biotype',
    'n_assemblies_assessed', 'n_assemblies_both', 'pct_assemblies_both',
    # Transcript concordance: Ensembl vs CAT agreement on the assembly
    'median_ens_to_cat_exact_pct', 'median_cat_to_ens_exact_pct',
    'median_ens_to_cat_concordance_rate', 'median_cat_to_ens_concordance_rate',
    # CDS integrity: Ensembl vs CAT agreement on the assembly (do they call the same CDS?)
    'n_assemblies_cds_assessed',
    'pct_start_codon_match', 'pct_stop_codon_match', 'pct_frameshift_detected',
    # Reference-anchored CDS quality: each annotator independently vs GRCh38
    'pct_ens_cds_match_ref', 'pct_cat_cds_match_ref',
    'pct_ens_coding_lost', 'pct_cat_coding_lost',
    'pct_ens_frameshift_ref', 'pct_cat_frameshift_ref',
    'predominant_ens_cds_change_type', 'predominant_cat_cds_change_type',
    # GRCh38 divergence: do both annotators agree, and in which direction?
    'predominant_divergence_category',
    'pct_both_agree_reference', 'pct_both_agree_diverged',
    'pct_cat_specific_divergence', 'pct_ensembl_specific_divergence',
    'pct_insufficient_data',
]
s1_col_order = [c for c in s1_col_order if c in s1_df.columns]
s1_df = s1_df[s1_col_order].sort_values('gene_name').reset_index(drop=True)

out_s1 = SUPP_DIR / 'supp_table_s1_gene_concordance_summary.tsv'
s1_df.to_csv(out_s1, sep='\t', index=False)
print(f"\nSaved S1: {out_s1}")
print(f"S1 shape: {s1_df.shape} ({time.time()-t0:.1f}s total)")
s1_df.head(3)

## Summary

## Build Supplementary Table S3: Discordant gene annotations

In [ ]:
t0 = time.time()

# S3: Genes where CAT and Ensembl give conflicting annotation signals.
#
# A gene is flagged as 'discordant' if ANY of the following hold across
# a meaningful number of assemblies:
#
#   divergence_discordant — in ≥10% of assessed assemblies one annotator diverges
#                           from GRCh38 while the other does not:
#                           cat_specific_divergence or ensembl_specific_divergence
#
#   cat_coding_lost       — CAT loses coding ability vs GRCh38 in ≥10% of assemblies
#                           (while Ensembl retains it); indicates CAT annotation fragility
#
#   cds_ref_discordant    — absolute gap in GRCh38 CDS match rate between the two
#                           annotators is ≥20 percentage points (protein-coding genes only)
#
# Genes present in fewer than 50 assemblies are excluded (insufficient power).

DISCORD_THRESHOLD = 10.0   # % assemblies showing discordance
CDS_DIFF_THRESHOLD = 20.0  # pp gap in reference CDS match rate
MIN_ASSEMBLIES = 50

s3_base = s1_df[s1_df['n_assemblies_assessed'] >= MIN_ASSEMBLIES].copy()

mask_div = (
    (s3_base['pct_cat_specific_divergence'].fillna(0) >= DISCORD_THRESHOLD) |
    (s3_base['pct_ensembl_specific_divergence'].fillna(0) >= DISCORD_THRESHOLD)
)

mask_cds_lost = s3_base['pct_cat_coding_lost'].fillna(0) >= DISCORD_THRESHOLD

mask_cds_ref = (
    s3_base['pct_ens_cds_match_ref'].notna() &
    s3_base['pct_cat_cds_match_ref'].notna() &
    ((s3_base['pct_ens_cds_match_ref'] - s3_base['pct_cat_cds_match_ref']).abs() >= CDS_DIFF_THRESHOLD)
)

s3_df = s3_base[mask_div | mask_cds_lost | mask_cds_ref].copy()

# Label the type(s) of discordance for each gene
def discordance_type(row):
    types = []
    if pd.notna(row.get('pct_cat_specific_divergence')) and row['pct_cat_specific_divergence'] >= DISCORD_THRESHOLD:
        types.append('cat_diverges')
    if pd.notna(row.get('pct_ensembl_specific_divergence')) and row['pct_ensembl_specific_divergence'] >= DISCORD_THRESHOLD:
        types.append('ensembl_diverges')
    if pd.notna(row.get('pct_cat_coding_lost')) and row['pct_cat_coding_lost'] >= DISCORD_THRESHOLD:
        types.append('cat_coding_lost')
    ens_ref = row.get('pct_ens_cds_match_ref')
    cat_ref = row.get('pct_cat_cds_match_ref')
    if pd.notna(ens_ref) and pd.notna(cat_ref) and abs(ens_ref - cat_ref) >= CDS_DIFF_THRESHOLD:
        types.append('cds_ref_discordant')
    return '|'.join(types)

s3_df.insert(2, 'discordance_type', s3_df.apply(discordance_type, axis=1))

# Sort: cat_specific_divergence first (most actionable), then cat_coding_lost
s3_df = s3_df.sort_values(
    ['pct_cat_specific_divergence', 'pct_cat_coding_lost'],
    ascending=False
).reset_index(drop=True)

out_s3 = SUPP_DIR / 'supp_table_s3_discordant_genes.tsv'
s3_df.to_csv(out_s3, sep='\t', index=False)
print(f"Saved S3: {out_s3}")
print(f"S3 shape: {s3_df.shape} ({time.time()-t0:.1f}s)")
print(f"\ndiscordance_type breakdown:")
print(s3_df['discordance_type'].value_counts().to_string())
print(f"\nbiotype breakdown:")
print(s3_df['ensembl_biotype'].value_counts().head(10).to_string())
s3_df[['gene_name', 'ensembl_biotype', 'discordance_type',
        'pct_cat_specific_divergence', 'pct_ensembl_specific_divergence',
        'pct_cat_coding_lost', 'pct_ens_cds_match_ref', 'pct_cat_cds_match_ref']].head(10)

In [ ]:
print(f"Table S1: {len(s1_df):,} genes × {len(s1_df.columns)} columns  (per-gene concordance summary)")
print(f"Table S2: {len(assm_summary):,} assemblies × {len(assm_summary.columns)} columns  (per-assembly summary)")
print(f"Table S3: {len(s3_df):,} genes × {len(s3_df.columns)} columns  (discordant annotations)")
print(f"\nS3 discordance breakdown:")
print(s3_df['discordance_type'].value_counts().to_string())
print(f"\nFiles written to: {SUPP_DIR}")
for f in sorted(SUPP_DIR.glob('*.tsv')):
    size = f.stat().st_size / 1024**2
    print(f"  {f.name}: {size:.1f} MB")